In [ ]:
import os

## Tracing ON!
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"]= "https://apac.api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "Your Key"         # LangSmith 플랫폼에서 발급한 키
os.environ["LANGSMITH_PROJECT"] = "langsmith"     # LangSmit 플랫폼에서 생성한 프로젝트
## Google Gemini 키
os.environ["GOOGLE_API_KEY"] = "Your Key" 

In [ ]:
## 문서를 다운 받은 위치
path = "./Docs/"

In [ ]:
## --------- Documnet Loader --------- ##
from langchain_community.document_loaders import TextLoader, PyPDFLoader, BSHTMLLoader

md_docs   = TextLoader(path+"sample_textbook.md", encoding="utf-8").load()
pdf_docs  = PyPDFLoader(path+"sample_pdf.pdf").load()
html_docs = BSHTMLLoader(path+"sample_homepage.html", open_encoding="utf-8").load()

## 문서리스트 하나로 통합
docs = md_docs + pdf_docs + html_docs
print(f"로딩된 Document 수: {len(docs)}")

/var/folders/r_/pvprk7vn6cq9ktv6k0gnx_240000gn/T/ipykernel_6281/1517601563.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader, BSHTMLLoader


로딩된 Document 수: 4


In [ ]:
## --------- Text Splitter --------- ##
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 500자 단위, 이전 청크의 50자 포함해서 청크 생성
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
## 청크(세분화 문서 리스트) 리스트 생성
split_docs = splitter.split_documents(docs)
print(f"분할된 chunk 수: {len(split_docs)}")

분할된 chunk 수: 20


In [ ]:
## --------- Embedding && VDB && Retriever --------- ##
##  문서 벡터화 (임베딩) -> 벡터 저장 (벡터 스토어) -> 검색(리트리버)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

## 제미나이 임베딩 모델 객체 선언
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"],
)
## Chroma VDB에 문서들(청크리스트)의 임베딩해서 저장
vectorstore = Chroma.from_documents(split_docs, embeddings)

## 검색시 Chroma VDB에서 상위 3개 관련문서 반환하도록 설정  
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
## --------- Prompt --------- ##
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "다음 문서를 근거로 사용자 질문에 답하세요. "
     "근거가 부족하면 '주어진 자료에서는 확인할 수 없습니다.'라고 답하세요.\n\n"
     "{context}"),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

def format_docs(ds):
    return "\n\n".join(d.page_content for d in ds)

rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
## 단일 짊문 테스트
print(rag.invoke("어댑터즈는 누가 운영하는 서비스인가요?"))

어댑터즈는 스타트업코드에서 운영하는 서비스입니다.


In [8]:
## 검증 질문
EVAL_QUESTIONS = [
    {
        "question": "어댑터즈는 누가 운영하는 서비스인가요?",
        "answer":   "어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.",
    },
    {
        "question": "5단 분석법은 어떤 다섯 단계로 구성되나요?",
        "answer":   "일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교 다섯 단계로 구성됩니다.",
    },
    {
        "question": "어댑터즈 AI 커리큘럼에는 몇 개의 교재가 있나요?",
        "answer":   "41개의 교재가 있습니다.",
    },
    {
        "question": "어댑터즈 교재는 얼마나 자주 업데이트되나요?",
        "answer":   "기존 교재는 분기별로 검토·업데이트되고, 신규 교재는 월 2~4개씩 추가됩니다.",
    },
    {
        "question": "어댑터즈는 일반 블로그와 어떤 점이 다른가요?",
        "answer":   "블로그는 정보가 파편적이지만, 어댑터즈는 5단 분석법으로 체계적으로 정리된 교재를 제공합니다.",
    },
]
print(f"검증 질문 수: {len(EVAL_QUESTIONS)}")

검증 질문 수: 5


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## prompt | model | StrOutput
chain = (
    ChatPromptTemplate.from_messages([
        ("system", "당신은 친절한 한국어 비서입니다."),
        ("human", "{question}"),
    ])
    | ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )
    | StrOutputParser()
)

answer = chain.invoke({"question": "LangSmith는 어떤 도구인가요?"})
print(answer)

LangSmith는 LangChain 프레임워크를 위한 **LLM(대규모 언어 모델) 애플리케이션 개발 및 디버깅 도구**입니다.

쉽게 말해, LangSmith는 LLM을 활용한 애플리케이션을 만들 때 발생하는 여러 어려움을 해결하고 개발 과정을 효율적으로 만들어주는 **"개발자 지원 도구"**라고 생각하시면 됩니다.

LangSmith가 제공하는 주요 기능들은 다음과 같습니다.

*   **트레이싱 (Tracing):** LLM 애플리케이션이 어떻게 작동하는지 단계별로 추적하고 시각화합니다. 어떤 프롬프트가 사용되었고, LLM이 어떤 응답을 했는지, 외부 도구를 호출했는지 등을 상세하게 볼 수 있어 문제 해결에 큰 도움을 줍니다.
*   **디버깅 (Debugging):** 트레이싱 정보를 바탕으로 애플리케이션의 오류를 쉽게 찾아내고 수정할 수 있습니다. 예상치 못한 결과가 나올 때 원인을 파악하기 용이합니다.
*   **평가 (Evaluation):** 개발한 LLM 애플리케이션의 성능을 자동으로 평가할 수 있는 기능을 제공합니다. 다양한 테스트 케이스를 설정하여 모델의 정확성, 관련성 등을 측정할 수 있습니다.
*   **모니터링 (Monitoring):** 실제 운영 환경에서 애플리케이션의 성능을 지속적으로 모니터링하고 문제가 발생했을 때 알림을 받을 수 있습니다.
*   **실험 관리 (Experiment Management):** 다양한 프롬프트, 모델, 파라미터 조합으로 실험을 진행하고 결과를 비교 분석하여 최적의 설정을 찾도록 돕습니다.

**LangSmith를 사용하면 좋은 점:**

*   LLM 애플리케이션 개발 및 디버깅 시간을 단축할 수 있습니다.
*   애플리케이션의 성능을 체계적으로 개선할 수 있습니다.
*   복잡한 LLM 애플리케이션의 동작 방식을 쉽게 이해하고 관리할 수 있습니다.
*   LangChain 프레임워크를 사용하는 개발자에게 특히 유용합니다.

요약하자면, LangSmith는 LLM 기반 애플리케이션을 **더 쉽고

In [ ]:
## ---------------------- <Tracing> ---------------------- ##
from langchain_core.runnables import RunnableConfig

## 필터링을 위해서 태그와 메타데이터 부착
rag.invoke(
    "어댑터즈는 누가 운영하는 서비스인가요?",
    config=RunnableConfig(
        tags=["version-2", "user-test"],                            ## 태그 부착
        metadata={"user_id": "u-42", "feature": "rag-demo"},        ## 메타데이터 부착
    ),
)

'어댑터즈는 스타트업코드에서 운영하는 서비스입니다.'

In [ ]:
## 일반 함수 추적을 위해서 @traceable 데코레이터 사용
from langsmith import traceable

@traceable(name="normalize_question")
def normalize(q: str) -> str:
    return q.strip().lower()

@traceable(name="rag_with_preprocess")
def run_pipeline(q: str) -> str:
    q = normalize(q)
    return rag.invoke(q)

print(run_pipeline("  어댑터즈는 누가 운영하는 서비스인가요?  "))

어댑터즈는 스타트업코드에서 운영하는 서비스입니다.


In [ ]:
## ---------------------- <Dataset> ---------------------- ##
from langsmith import Client

# LangSmith Client 생성
client = Client()

print("LangSmith Client 준비 완료")
print("Project:", os.environ["LANGSMITH_PROJECT"])

LangSmith Client 준비 완료
Project: langsmith


In [ ]:
DATASET_NAME = os.environ["LANGSMITH_PROJECT"]

# 중복 데이터셋 존재 체크
existing = [d for d in client.list_datasets(dataset_name=DATASET_NAME)]

if existing:
    dataset = existing[0]
    print(f"기존 Dataset 사용: {dataset.id}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="어댑터즈 RAG 답변 품질 평가용",
    )
    print(f"새 Dataset 생성: {dataset.id}")

기존 Dataset 사용: ea92b7fc-8e7a-428c-af83-52dd60ffe1c3


In [ ]:
## 데이터셋의 입력 출력 각 형태에 맞게 리스트 생성 
inputs  = [{"question": ex["question"]} for ex in EVAL_QUESTIONS]
outputs = [{"answer":   ex["answer"]}   for ex in EVAL_QUESTIONS]

client.create_examples(
    dataset_id=dataset.id,
    inputs=inputs,
    outputs=outputs,
)

print(f"Example {len(EVAL_QUESTIONS)}건 추가 완료")

Example 5건 추가 완료


In [ ]:
## 정상으로 데이터셋 불러왔는지 확인
loaded = client.read_dataset(dataset_name=DATASET_NAME)

examples = list(client.list_examples(dataset_id=loaded.id))
print(f"총 Example 수: {len(examples)}")

for ex in examples[:3]:
    print("Q:", ex.inputs["question"])
    print("A:", ex.outputs["answer"] if ex.outputs else "(없음)")
    print()

총 Example 수: 5
Q: 어댑터즈는 누가 운영하는 서비스인가요?
A: 어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.

Q: 어댑터즈는 일반 블로그와 어떤 점이 다른가요?
A: 블로그는 정보가 파편적이지만, 어댑터즈는 5단 분석법으로 체계적으로 정리된 교재를 제공합니다.

Q: 5단 분석법은 어떤 다섯 단계로 구성되나요?
A: 일반 명사, 고유 명사, 사용 이유, 사용 방법, 다른 기술과의 비교 다섯 단계로 구성됩니다.



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableConfig

# 실패 케이스 재현용 약한 프롬프트
# 운영용으로 사용하면 젇래 안됨!!
weak_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 어댑터즈 홈페이지의 마케팅 문구를 작성하는 담당자입니다. "
     "다음 문서를 참고하되, 문서에 직접 없는 정책도 사용자가 안심할 수 있도록 자연스럽게 보완해 답변하세요. "
     "'확인할 수 없습니다'라는 표현은 사용하지 말고, FAQ 형식으로 단정적인 안내 문구를 작성하세요.\n\n"
     "{context}"),
    ("human", "{question}"),
])

weak_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | weak_prompt
    | llm
    | StrOutputParser()
)

bad_case_question = "어댑터즈의 취업 보장과 환불 규정을 FAQ 형식으로 정리해 주세요."

answer = weak_rag.invoke(
    bad_case_question,
    config=RunnableConfig(
        tags=["dataset-candidate", "bad-case", "weak-prompt"],
        metadata={
            "case_type": "unsupported_policy_generation",
            "source": "manual_review",
        },
    ),
)

print("질문:", bad_case_question)
print("답변:", answer)

질문: 어댑터즈의 취업 보장과 환불 규정을 FAQ 형식으로 정리해 주세요.
답변: ## 어댑터즈 자주 묻는 질문 (FAQ)

**Q1. 어댑터즈를 통해 취업이 보장되나요?**

A1. 어댑터즈는 실무 중심의 커리큘럼을 통해 학습자의 역량 강화에 집중합니다. 이를 바탕으로 개인의 노력과 성장에 따라 성공적인 취업을 이루도록 지원하지만, 교육 과정 수료만으로 취업을 100% 보장하지는 않습니다. 어댑터즈는 학습자가 실제 현장에서 요구되는 기술을 습득하고 경쟁력을 갖출 수 있도록 최선을 다해 지원합니다.

**Q2. 환불 규정이 궁금합니다.**

A2. 어댑터즈는 학습자의 만족도를 최우선으로 생각하며, 합리적인 환불 정책을 운영하고 있습니다.

*   **결제 후 7일 이내, 학습 시작 전:** 전액 환불이 가능합니다.
*   **결제 후 7일 이내, 학습 일부 시작 후:** 학습 진도율에 따라 부분 환불이 적용됩니다.
*   **결제 후 7일 이후:** 환불이 어려운 점 양해 부탁드립니다.

보다 자세한 환불 규정 및 절차는 [환불 규정 페이지 링크 삽입]에서 확인하실 수 있습니다.


In [ ]:
## ---------------------- <Evaluator> ---------------------- ##

assert "rag" in dir(),  "베이스라인을 먼저 실행하세요"
assert "llm" in dir(),  "베이스라인을 먼저 실행하세요"

examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f"Example 수: {len(examples)}")

Example 수: 5


In [21]:
def target(inputs):
    return {"answer": rag.invoke(inputs["question"])}

In [22]:


## 휴리스틱 방식
def contains_expected_keyword(run, example):
    pred = run.outputs.get("answer", "")
    expected = example.outputs.get("answer", "")

    # === 기대 답변에서 명사로 보이는 단어 한두 개를 키워드로 사용 ===
    keywords = [w for w in expected.split() if len(w) >= 2][:2]
    hit = all(k in pred for k in keywords)

    return {
        "key": "contains_expected_keyword",
        "score": 1 if hit else 0,
        "comment": f"필수 키워드 {keywords} 포함 여부",
    }

## LLM 판단 방식
JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 답변 품질을 평가하는 채점자입니다.\n"
     "아래 기대 답변(reference)과 모델 답변(prediction)을 비교하고,\n"
     "의미가 일치하면 1, 부분적으로만 일치하면 0.5, 무관하면 0을 점수로 매기세요.\n"
     "응답은 반드시 첫 줄에 0/0.5/1 중 하나의 숫자만, 둘째 줄부터 짧은 이유를 적으세요."),
    ("human",
     "질문: {question}\n\n"
     "기대 답변: {reference}\n\n"
     "모델 답변: {prediction}"),
])

judge_chain = JUDGE_PROMPT | llm | StrOutputParser()

def llm_judge(run, example):
    reply = judge_chain.invoke({
        "question": example.inputs["question"],
        "reference": example.outputs["answer"],
        "prediction": run.outputs["answer"],
    })
    # === 첫 줄의 숫자만 점수로 사용 ===
    first_line = reply.strip().splitlines()[0].strip()
    try:
        score = float(first_line)
    except ValueError:
        score = 0
    return {
        "key": "llm_judge_semantic_match",
        "score": score,
        "comment": reply,
    }


In [ ]:
from langsmith.evaluation import evaluate

## 대시보드에서 각 experiment 결과 확인
result = evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[contains_expected_keyword, llm_judge],
    experiment_prefix="v1-baseline",
)

print(result)

/Users/emet/MyRagProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'v1-baseline-de57047c' at:
https://apac.smith.langchain.com/o/1fb3aa0d-b432-4861-94f4-4941059e9da3/datasets/ea92b7fc-8e7a-428c-af83-52dd60ffe1c3/compare?selectedSessions=22439778-76a7-4fa2-81b4-d2de06dda59a




0it [00:00, ?it/s]Error running target function: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Traceback (most recent call last):
  File "/Users/emet/MyRagProject/.venv/lib/python3.14/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/r_/pvprk7vn6cq9ktv6k0gnx_240000gn/T/ipykernel_6281/1624704168.py", line 2, in target
    return {"answer": rag.invoke(inputs["question"])}
                      ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/Users/emet/MyRagProject/.venv/lib/python3.14/site-packages/langchain_core/runnables/base.py", line 3444, in invoke
    input_ = context.run(step.invoke, input_, config)
  File "/Users/emet/MyRagProject/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py", lin

<ExperimentResults v1-baseline-de57047c>


In [ ]:
## ---------------------- <Prompt Management> ---------------------- ##
from langchain_core.prompts import ChatPromptTemplate

## prompt_v1 프롬프트 생성
prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 답하세요.\n\n{context}"),
    ("human", "{question}"),
])

## 프롬프트를 허브에 push
result = client.push_prompt(
    "adapterz-rag-prompt",
    object=prompt_v1,
    description="어댑터즈 RAG 답변 프롬프트 (baseline)",
)
print(result)

https://apac.smith.langchain.com/prompts/adapterz-rag-prompt/a572c07d?organizationId=1fb3aa0d-b432-4861-94f4-4941059e9da3


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Hub에서 프롬프트 pull
hub_prompt = client.pull_prompt("adapterz-rag-prompt")

## pull prompt 체인 연결
rag_hub = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | hub_prompt
    | llm
    | StrOutputParser()
)



print(rag_hub.invoke("어댑터즈는 누가 운영하는 서비스인가요?"))

In [25]:
## 새버전의 프롬프트 생성
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 어댑터즈 서비스에 익숙한 한국어 도우미입니다.\n"
     "아래 문서만 근거로 한 단락으로 간결하게 답하세요.\n\n{context}"),
    ("human", "{question}"),
])

client.push_prompt(
    "adapterz-rag-prompt",
    object=prompt_v2,
    description="간결한 한 단락 답변으로 개선",
)

'https://apac.smith.langchain.com/prompts/adapterz-rag-prompt/20a1d245?organizationId=1fb3aa0d-b432-4861-94f4-4941059e9da3'

In [ ]:
# 코드에서는 특정 버전을 직접 지정해 가져올 수 있음 ===
latest_prompt = client.pull_prompt("adapterz-rag-prompt")

# Hub에서 확인한 특정 commit hash를 직접 지정
specific_prompt = client.pull_prompt("adapterz-rag-prompt:<commit-hash>")

# Hub에서 production 태그를 설정한 경우 태그 기준으로 가져오기
prod_prompt = client.pull_prompt("adapterz-rag-prompt:production")